In [ ]:
!pip install kafka-python boto3 pyarrow pandas

In [ ]:
#!/usr/bin/env python3
# vehicle_gps_producer.py
import json, time, random, boto3, pyarrow.parquet as pq
import pyarrow.fs as fs
from kafka import KafkaProducer
from datetime import datetime
import re

# ========== CONFIGURATION ==========
KAFKA_BROKER   = "kafka:9092"
TOPIC_GPS      = "raw.gps"
MINIO_ENDPOINT = "http://100.104.4.32:9000"   # IP Tailscale de votre camarade
MINIO_KEY      = "minioadmin"
MINIO_SECRET   = "minioadmin123"
BUCKET         = "curated"
PREFIX         = "porto-trajectories/"
SPEED_FACTOR   = 10          # x10 plus rapide
GPS_INTERVAL   = 15          # secondes réelles entre deux pings
NOISE_SIGMA    = 0.0002      # ±20 m
BLACKOUT_PROB  = 0.05

# ========== KAFKA PRODUCER ==========
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
    acks="all"
)
print("✅ Kafka connecté")

# ========== CONNEXION MINIO ==========
s3 = boto3.client("s3", endpoint_url=MINIO_ENDPOINT,
                  aws_access_key_id=MINIO_KEY, aws_secret_access_key=MINIO_SECRET)
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
parquet_files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"📁 {len(parquet_files)} fichiers Parquet trouvés")

arrow_fs = fs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT,
    access_key=MINIO_KEY, secret_key=MINIO_SECRET, scheme="http"
)

# ========== GÉNÉRATEUR DE TRAJETS (streaming) ==========
def iter_trajectories():
    for file_key in parquet_files:
        full_path = f"{BUCKET}/{file_key}"
        table = pq.read_table(full_path, filesystem=arrow_fs)
        cols = table.column_names
        needed = ["casa_origin_lon", "casa_origin_lat", "casa_dest_lon", "casa_dest_lat",
                  "TAXI_ID", "trip_duration_min"]
        for col in needed:
            if col not in cols:
                raise ValueError(f"Colonne {col} manquante dans {file_key}")
        # Extraire origin_zone_id depuis le chemin
        match = re.search(r"origin_zone_id=(\d+)", file_key)
        zone_id = int(match.group(1)) if match else 0
        data = {col: table.column(col).to_pylist() for col in needed}
        for i in range(table.num_rows):
            yield {
                "TAXI_ID": data["TAXI_ID"][i],
                "casa_origin_lon": data["casa_origin_lon"][i],
                "casa_origin_lat": data["casa_origin_lat"][i],
                "casa_dest_lon": data["casa_dest_lon"][i],
                "casa_dest_lat": data["casa_dest_lat"][i],
                "trip_duration_min": data["trip_duration_min"][i],
                "origin_zone_id": zone_id
            }

# ========== FONCTIONS ==========
def add_noise(lat, lon):
    return lat + random.gauss(0, NOISE_SIGMA), lon + random.gauss(0, NOISE_SIGMA)

def simulate_trip(row):
    taxi_id = int(row["TAXI_ID"])
    o_lon, o_lat = row["casa_origin_lon"], row["casa_origin_lat"]
    d_lon, d_lat = row["casa_dest_lon"], row["casa_dest_lat"]
    duration_min = float(row["trip_duration_min"])
    zone_id = row["origin_zone_id"]

    # Nombre de points (un point toutes les 15s → 4 points par minute)
    n_points = max(4, int(duration_min * 4))

    for i in range(n_points):
        if random.random() < BLACKOUT_PROB:
            delay = random.uniform(60, 180)
            time.sleep(delay / SPEED_FACTOR)
            continue

        ratio = i / (n_points - 1) if n_points > 1 else 0
        lon = o_lon + (d_lon - o_lon) * ratio
        lat = o_lat + (d_lat - o_lat) * ratio
        # Ajouter bruit GPS
        lat_noisy, lon_noisy = add_noise(lat, lon)

        event = {
            "taxi_id": taxi_id,
            "timestamp": int(datetime.now().timestamp()),
            "lat": round(lat_noisy, 6),
            "lon": round(lon_noisy, 6),
            "speed": round(random.uniform(10, 70), 1),
            "status": "available" if i == 0 else "on_trip",
            "zone_id": zone_id
        }
        producer.send(TOPIC_GPS, key=str(taxi_id), value=event)
        time.sleep(GPS_INTERVAL / SPEED_FACTOR)

    # Dernier message (arrivée)
    producer.send(TOPIC_GPS, key=str(taxi_id), value={
        "taxi_id": taxi_id,
        "timestamp": int(datetime.now().timestamp()),
        "lat": round(add_noise(d_lat, d_lon)[0], 6),
        "lon": round(add_noise(d_lat, d_lon)[1], 6),
        "speed": 0.0,
        "status": "available",
        "zone_id": zone_id
    })

# ========== BOUCLE PRINCIPALE ==========
print("🚕 GPS Producer démarré")
print(f"   Topic : {TOPIC_GPS} | Vitesse x{SPEED_FACTOR}")
count = 0
try:
    while True:
        for row in iter_trajectories():
            simulate_trip(row)
            count += 1
            if count % 20 == 0:
                producer.flush()
                print(f"   ✅ {count} trajets GPS envoyés")
        print("🔄 Replay complet des données...")
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} trajets envoyés")
    producer.flush()
    producer.close()

In [ ]:
# ========== BOUCLE PRINCIPALE ==========
print("🚕 GPS Producer démarré")
print(f"   Topic : {TOPIC_GPS} | Vitesse x{SPEED_FACTOR}")
count = 0
try:
    while True:
        for row in iter_trajectories():
            simulate_trip(row)
            count += 1
            if count % 20 == 0:
                producer.flush()
                print(f"   ✅ {count} trajets GPS envoyés")
        print("🔄 Replay complet des données...")
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} trajets envoyés")
    producer.flush()
    producer.close()

In [ ]:
gps.terminate()